In [1]:
import sys
sys.path.append('/kaggle/input/datasets/pankajdeopa/vr-project-config/')
import config

Device : cpu
Save dir: /kaggle/working/checkpoints/


In [2]:
import os, json

DATA_ROOT = '/kaggle/input/datasets/varun000reddy/'

TRAIN_IMG = DATA_ROOT + 'training/train/image/'
TRAIN_ANN = DATA_ROOT + 'training/train/annos/'

VAL_IMG   = DATA_ROOT + 'validation/validation/image/'
VAL_ANN   = DATA_ROOT + 'validation/validation/annos/'

TEST_IMG  = DATA_ROOT + 'testing/test/test/'
TEST_ANN  = DATA_ROOT + 'testing/test/json_for_test/'

# Verify sample annotation
sample_id = os.listdir(TRAIN_ANN)[0].replace('.json', '')
print("Sample ID:", sample_id)

with open(f'{TRAIN_ANN}/{sample_id}.json') as f:
    ann = json.load(f)

print(json.dumps(ann, indent=2))

Sample ID: 039350
{
  "item2": {
    "segmentation": [
      [
        360,
        172,
        364,
        166,
        371,
        163,
        325,
        168,
        299,
        202,
        279,
        234,
        258,
        263,
        233,
        299,
        256,
        311,
        273,
        273,
        288,
        246,
        304,
        223,
        308,
        203,
        298,
        211,
        279,
        239,
        258,
        269,
        287,
        292,
        306,
        242,
        341,
        213,
        360,
        172
      ],
      [
        410,
        182,
        409,
        188,
        406,
        194,
        383,
        230,
        380,
        277,
        366,
        323,
        402,
        330,
        414,
        291,
        424,
        250,
        424,
        240,
        418,
        269,
        409,
        299,
        397,
        336,
        385,
        367,
        418,
        375,
        422

In [3]:
import os

for split, inner in [('training', 'train'), ('testing', 'test'), ('validation', 'validation')]:
    path = os.path.join(DATA_ROOT, split, inner)
    print(f"{split}/{inner}: {os.listdir(path)}")

training/train: ['annos', 'image']
testing/test: ['json_for_test', 'test']
validation/validation: ['annos', 'image']


In [4]:
from collections import Counter

category_counter = Counter()

ann_files = os.listdir(TRAIN_ANN)
print(f"Total training annotations: {len(ann_files)}")

for fname in ann_files:
    with open(os.path.join(TRAIN_ANN, fname)) as f:
        ann = json.load(f)
    
    for key, item in ann.items():
        # Skip non-item keys like 'source' and 'pair_id'
        if not key.startswith('item'):
            continue
        cat_id   = item['category_id']
        cat_name = item['category_name']
        category_counter[(cat_id, cat_name)] += 1

# Display all categories sorted by frequency
print("\nCategory frequencies:")
for (cat_id, cat_name), count in category_counter.most_common():
    print(f"  [{cat_id}] {cat_name:<25} : {count}")

# Top-5
top5 = category_counter.most_common(5)
print("\nTop-5 categories selected:")
for (cat_id, cat_name), count in top5:
    print(f"  [{cat_id}] {cat_name} — {count} instances")

Total training annotations: 191961

Category frequencies:
  [1] short sleeve top          : 71645
  [8] trousers                  : 55387
  [7] shorts                    : 36616
  [2] long sleeve top           : 36064
  [9] skirt                     : 30835
  [12] vest dress                : 17949
  [10] short sleeve dress        : 17211
  [5] vest                      : 16095
  [4] long sleeve outwear       : 13457
  [11] long sleeve dress         : 7907
  [13] sling dress               : 6492
  [6] sling                     : 1985
  [3] short sleeve outwear      : 543

Top-5 categories selected:
  [1] short sleeve top — 71645 instances
  [8] trousers — 55387 instances
  [7] shorts — 36616 instances
  [2] long sleeve top — 36064 instances
  [9] skirt — 30835 instances


In [5]:
import pandas as pd
from PIL import Image

# Top-5 category mapping: category_id -> class index (0-4)
TOP5_CATEGORIES = {1: 0, 8: 1, 7: 2, 2: 3, 9: 4}
IDX_TO_NAME = {0: 'short_sleeve_top', 1: 'trousers', 2: 'shorts', 3: 'long_sleeve_top', 4: 'skirt'}

def build_labels(ann_dir, img_dir, split_name):
    records = []
    skipped = 0

    for fname in os.listdir(ann_dir):
        if not fname.endswith('.json'):
            continue

        img_id = fname.replace('.json', '')
        img_path = os.path.join(img_dir, img_id + '.jpg')

        # Skip if image missing
        if not os.path.exists(img_path):
            skipped += 1
            continue

        with open(os.path.join(ann_dir, fname)) as f:
            ann = json.load(f)

        # Build multi-hot vector
        label = [0, 0, 0, 0, 0]
        for key, item in ann.items():
            if not key.startswith('item'):
                continue
            cat_id = item['category_id']
            if cat_id in TOP5_CATEGORIES:
                label[TOP5_CATEGORIES[cat_id]] = 1

        # Skip images with no top-5 categories
        if sum(label) == 0:
            skipped += 1
            continue

        records.append({
            'image_path': img_path,
            'label_0': label[0],  # short_sleeve_top
            'label_1': label[1],  # trousers
            'label_2': label[2],  # shorts
            'label_3': label[3],  # long_sleeve_top
            'label_4': label[4],  # skirt
            'split': split_name
        })

    print(f"{split_name}: {len(records)} kept, {skipped} skipped")
    return records

# Build for all splits
all_records = []
all_records += build_labels(TRAIN_ANN, TRAIN_IMG, 'train')
all_records += build_labels(VAL_ANN,   VAL_IMG,   'val')

# Save master CSV
df = pd.DataFrame(all_records)
df.to_csv('/kaggle/working/master_labels.csv', index=False)

print(f"\nTotal records saved: {len(df)}")
print(df.head())
print("\nLabel distribution:")
for i, name in IDX_TO_NAME.items():
    print(f"  {name}: {df[f'label_{i}'].sum()} positive samples")

train: 144174 kept, 47787 skipped
val: 23741 kept, 8412 skipped

Total records saved: 167915
                                          image_path  label_0  label_1  \
0  /kaggle/input/datasets/varun000reddy/training/...        0        1   
1  /kaggle/input/datasets/varun000reddy/training/...        1        0   
2  /kaggle/input/datasets/varun000reddy/training/...        0        0   
3  /kaggle/input/datasets/varun000reddy/training/...        1        0   
4  /kaggle/input/datasets/varun000reddy/training/...        1        0   

   label_2  label_3  label_4  split  
0        0        0        0  train  
1        0        0        1  train  
2        1        0        0  train  
3        0        0        0  train  
4        0        0        1  train  

Label distribution:
  short_sleeve_top: 82957 positive samples
  trousers: 64463 positive samples
  shorts: 40465 positive samples
  long_sleeve_top: 41667 positive samples
  skirt: 37124 positive samples


In [6]:

#compute Compute pos_weight for class imbalance
import torch
import pandas as pd

df = pd.read_csv('/kaggle/working/master_labels.csv')
train_df = df[df['split'] == 'train']

label_cols = ['label_0', 'label_1', 'label_2', 'label_3', 'label_4']
labels = train_df[label_cols].values

N = len(train_df)
pos_counts = labels.sum(axis=0)
neg_counts = N - pos_counts
pos_weight = neg_counts / pos_counts

print(f"Total train samples: {N}")
print("\nper-class pos_weight:")
for i, name in IDX_TO_NAME.items():
    print(f"  {name:<20} : {pos_weight[i]:.3f}")

# Save as tensor for training
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32)
print("\npos_weight tensor:", pos_weight_tensor)

Total train samples: 144174

per-class pos_weight:
  short_sleeve_top     : 1.043
  trousers             : 1.623
  shorts               : 2.968
  long_sleeve_top      : 3.033
  skirt                : 3.708

pos_weight tensor: tensor([1.0425, 1.6228, 2.9682, 3.0327, 3.7077])


In [7]:
import torch

pos_weight_tensor = torch.tensor([1.0425, 1.6228, 2.9682, 3.0327, 3.7077], dtype=torch.float32)
torch.save(pos_weight_tensor, '/kaggle/working/pos_weight.pt')
print("Saved pos_weight.pt")

Saved pos_weight.pt


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np

# ── Dataset Class ──────────────────────────────────────────────
class ApparelDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df         = df.reset_index(drop=True)
        self.transform  = transform
        self.label_cols = ['label_0','label_1','label_2','label_3','label_4']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row[self.label_cols].values.astype(np.float32))
        return img, label


# ── Transforms ─────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


# ── Load CSV & Create Datasets ──────────────────────────────────
df       = pd.read_csv('/kaggle/working/master_labels.csv')
train_df = df[df['split'] == 'train']
val_df   = df[df['split'] == 'val']

train_dataset = ApparelDataset(train_df, transform=train_transform)
val_dataset   = ApparelDataset(val_df,   transform=val_transform)


# ── Create DataLoaders ──────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size  = 32,
    shuffle     = True,
    num_workers = 2,
    pin_memory  = False
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = 32,
    shuffle     = False,
    num_workers = 2,
    pin_memory  = False
)


# ── Sanity Check ────────────────────────────────────────────────
images, labels = next(iter(train_loader))
print(f"Train dataset size : {len(train_dataset)}")
print(f"Val dataset size   : {len(val_dataset)}")
print(f"Image batch shape  : {images.shape}")   # expected: [32, 3, 224, 224]
print(f"Label batch shape  : {labels.shape}")   # expected: [32, 5]
print(f"Sample label vector: {labels[0]}")

Train dataset size : 144174
Val dataset size   : 23741
Image batch shape  : torch.Size([32, 3, 224, 224])
Label batch shape  : torch.Size([32, 5])
Sample label vector: tensor([0., 0., 1., 1., 0.])
